In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
plt.rcParams["animation.html"]="jshtml"

List_methode_de_flux= ["primitive","conservative"]
List_condition_limite=["Neumann","Dirichlet","Periodique"]

In [125]:
class Finite_Volume:


    def __init__(self,K_l,K_r, flux_methode, film_bool =False, condition_limite = "Neumann",  ghost_nodes=[0,0], domaine=(0.,1.) ):
        Valid_init = True
        self.flux = lambda u,v,xi,xj :0*u+0*v
        self.kl= K_l; self.kr =K_r

        in_list =False
        for methode in List_methode_de_flux:
            if(flux_methode ==methode):  in_list =True
        if(not in_list): Valid_init = False

        in_list =False
        for condition in List_condition_limite:
            if(condition_limite ==condition): in_list =True
        if(not in_list): Valid_init = False
        
    
        self.flux_methode = flux_methode;   self.condition_limite = condition_limite
        self.ghost_nodes=ghost_nodes
        self.film_bool = film_bool
        self.domaine= domaine
        if(not Valid_init): f"Methode non reconnue"

    def flux_function(self,dx):
        # k = lambda x: self.kl*(x<-dx/2) + self.kr*(x>dx/2) + (x<dx/2)*(x>-dx/2)*( abs(x-dx/2)*self.kl + abs(x+dx/2)*self.kr)
        k = lambda x: self.kl*(x<0) + self.kr*(x>0)
        self.k = k
        if(self.flux_methode == "primitive"):
            self.flux= lambda u,v,xi,xj : 2*k(xi)*k(xj)/(k(xi)+k(xj)) * u*(1-v)/(u+(1-v))
        elif(self.flux_methode =="conservative"):
            self.flux= lambda u,v,xi,xj : (k(xi)*u * k(xj)*(1-v) )/(k(xi)*u + k(xj)*(1-v)) 


    def sol(self, u0, J,T,  CFL=0.5, nb_images=30):
        dx = 1./J
        
        a= self.domaine[0]; b= self.domaine[1]

        X = np.linspace(a+dx,b,J) -dx/2


        f = lambda u,x: self.k(x)*u*(1-u) ; 
        flux = self.flux
        U = u0(X)

        FG = np.empty_like(X) 
        FD = np.empty_like(X)

        
        U_sol = []; dt_list=[]
        vitesse=0

        t=0.; n =0; dt=0.
        while t<T:
            #calcul de dt

            vitesse = max(vitesse, max(self.k(X)*(1-2*U)))

            if(vitesse !=0) : dt = (min(CFL * dx /(2* vitesse), T-t))
            else :            dt = CFL*dx  

            if(dt<0): dt = 10e-10; print(f"!!!!!!!!!!!!!!!!! dt<0 !!!!!!!!!")

            ## Calcul du flux
            for j in range(len(X)):
                
                if(j==0):   
                    if(self.condition_limite =="Dirichlet"):    FG[j] = flux(self.ghost_nodes[0],U[j],a,a+dx )
                    elif(self.condition_limite =="Neumann"):    FG[j] = flux(U[j]+self.ghost_nodes[0],U[j],a,a+dx)
                    elif(self.condition_limite =="Periodique"): FG[j] = flux(U[-1], U[j],b,a)

                else :      FG[j]= flux(U[j-1],U[j],(j-1)*dx,j*dx); 
                
                if(j==J-1):   
                    if(self.condition_limite =="Dirichlet"):    FD[j] = flux(U[j], self.ghost_nodes[1],b,b+dx)
                    elif(self.condition_limite =="Neumann"):    FD[j] = flux(U[j], U[j]+self.ghost_nodes[1],b,b+dx)
                    elif(self.condition_limite =="Periodique"): FD[j] = flux(U[j], U[0],b,a)

                else :      FD[j]= flux(U[j],U[j+1],j*dx,(j+1)*dx) 

            # print(max(FD), max)

            ## eval
            U_sol.append(U.copy())

            ## Calcul de la solution
            for j in range(len(X)):
                U[j] = U[j] -(dt/dx)* (FD[j]-FG[j])

            n+=1; t+= dt; dt_list.append(dt)
            if n%10 :print("------ ---- t= ",t," -----------------")

        U_anime = []; nb_iter = min(len(U_sol)-1, nb_images)
        for i in np.linspace(0,len(U_sol)-1,nb_iter): U_anime.append(U_sol[int(i)])
        self.anime = U_anime; self.dt_list = dt_list

    def show_sol(self,bound=[0,0]):

        lenght_animation = len(self.anime); T= sum(self.dt_list); L=self.domaine[1]-self.domaine[0]
        print(lenght_animation)

        J = len(self.anime[0]); 
        dx = (self.domaine[1]- self.domaine[0])/J

        X= np.linspace(self.domaine[0],self.domaine[1],J,endpoint=False) -dx/2

        fig, (ax , ax_film) = plt.subplots(1,2)
        
        imag_xt =  np.zeros((lenght_animation,J))
        for i in range(lenght_animation) : imag_xt[i,:] = self.anime[lenght_animation -i-1]
        im = ax_film.imshow(imag_xt,extent=((self.domaine[0], self.domaine[1], 0,T)))
        

        fig.colorbar(im, ax=ax_film, label='Interactive colorbar')
        ax_film.set_aspect(L/T)
        ax_film.set_ylabel("t")
        ax_film.set_xlabel("x")

        line = ax_film.plot(X, 0*X,'r')[0]

        m = min(np.min(self.anime)*0.9,bound[0]); M = max(np.max(self.anime)*1.1,bound[1])

        ax.set(xlim=[self.domaine[0],self.domaine[1]], ylim=[m,M], xlabel='x', ylabel='U')

        u_plot = ax.plot(X, self.anime[0],'b', label="u")[0]
        ku_plot= ax.plot(X, self.k(X)*self.anime[0]*(1-self.anime[0]),'r',label="ku(1-u)")[0]

        ax.legend(fancybox=True, shadow=True, loc='upper left')     


        def update(i):
            t= sum(self.dt_list[:i])
            print(i+1,"/",lenght_animation)
            # IPython.display.clear_output(wait=True)
            u_plot.set_ydata(self.anime[i])
            ku_plot.set_ydata(self.k(X)*self.anime[i]*(1-self.anime[i]))
            line.set_ydata(i*T/(lenght_animation) +0*X)
            

        anime = animation.FuncAnimation(fig=fig, func=update, frames=lenght_animation)
        return anime
    

In [126]:
#
ul=0.5;ur=0.3
U0  = lambda x: ul*(x<0.) + ur*(x>=0.)

Fv = Finite_Volume(K_l=2,K_r=1 ,flux_methode="primitive",film_bool=True, condition_limite="Neumann", domaine=[-5,5])
J=100
Fv.flux_function(dx=1/J)
Fv.sol(U0,J=J,T=4,CFL=0.45)
anime =Fv.show_sol(bound=[0,1])
plt.close()
anime

------ ---- t=  0.005625000000000001  -----------------
------ ---- t=  0.009625000000000002  -----------------
------ ---- t=  0.012133234101849508  -----------------
------ ---- t=  0.014221616718472534  -----------------
------ ---- t=  0.016078259204158108  -----------------
------ ---- t=  0.017783336249207003  -----------------
------ ---- t=  0.01938040661420936  -----------------
------ ---- t=  0.02089638871800791  -----------------
------ ---- t=  0.022349324570456807  -----------------
------ ---- t=  0.02511386588865363  -----------------
------ ---- t=  0.026442103834067907  -----------------
------ ---- t=  0.027742325157195265  -----------------
------ ---- t=  0.02901898912511614  -----------------
------ ---- t=  0.03027569674398932  -----------------
------ ---- t=  0.03151539332616787  -----------------
------ ---- t=  0.03274051366680354  -----------------
------ ---- t=  0.03395308867158069  -----------------
------ ---- t=  0.03515482533249917  -----------------
-

In [117]:
#
ul=0.95;ur=0.8
U0  = lambda x: ul*(x<0.) + ur*(x>=0.)

Fv = Finite_Volume(K_l=2,K_r=1 ,flux_methode="conservative",film_bool=True, condition_limite="Neumann", domaine=[-5,5])

J=100
Fv.flux_function(dx=1/J)
Fv.sol(U0,J=J,T=2,CFL=0.45)
anime =Fv.show_sol(bound=[0,1])
plt.close()
anime

------ ---- t=  0.0045000000000000005  -----------------
------ ---- t=  0.009000000000000001  -----------------
------ ---- t=  0.013500000000000002  -----------------
------ ---- t=  0.018000000000000002  -----------------
------ ---- t=  0.022500000000000003  -----------------
------ ---- t=  0.027000000000000003  -----------------
------ ---- t=  0.0315  -----------------
------ ---- t=  0.036000000000000004  -----------------
------ ---- t=  0.04050000000000001  -----------------
------ ---- t=  0.049500000000000016  -----------------
------ ---- t=  0.05400000000000002  -----------------
------ ---- t=  0.058500000000000024  -----------------
------ ---- t=  0.06300000000000003  -----------------
------ ---- t=  0.06750000000000003  -----------------
------ ---- t=  0.07200000000000004  -----------------
------ ---- t=  0.07650000000000004  -----------------
------ ---- t=  0.08100000000000004  -----------------
------ ---- t=  0.08550000000000005  -----------------
------ ---- t